# Single-emitter ESM observation series generation

This notebook demonstrates generation and saving of synthetic single-aircraft emitter tracks for the experiments described in `docs/single_emitter_observation_series.md`.

Series entries may now contain multiple in-track radar-mode switches. Per-observation ground-truth labels and the aggregate truth mode sequence are retained for evaluation, but the inference view strips those labels before downstream scoring/classification to avoid leakage.

In [1]:
from pathlib import Path
import json

from esm_observation_series_generator import generate_observation_series, observations_without_ground_truth

In [2]:
data = generate_observation_series(
    count=25,
    seed=42,
    min_duration_s=1.0,
    max_duration_s=60.0,
    sample_interval_s=0.5,
    mode_switch_probability=0.08,  # sampled between adjacent observations; entries can switch several times
)

len(data["observation_series"]), data["metadata"]

(25,
 {'schema_version': '1.0',
  'series_count': 25,
  'default_count': 2500,
  'seed': 42,
  'min_duration_s': 1.0,
  'max_duration_s': 60.0,
  'sample_interval_s': 0.5,
  'mode_switch_probability': 0.08,
  'max_mode_switches': None,
  'generated_at': '2026-07-09T15:04:35.303508Z'})

In [3]:
data["observation_series"][0]

{'series_id': 'esm_series_00001',
 'emitter_type': 'aircraft',
 'sample_interval_s': 0.5,
 'duration_s': 44.988,
 'observation_count': 91,
 'mode_shift_sequence_indices': [7, 10, 17, 39, 42, 65, 87],
 'mode_shift_sequence_index': 7,
 'ground_truth_mode_sequence': [{'sequence_index': 0,
   'mode': 'track_while_scan',
   'mode_id': 'radar_mode:an_apg_68:track_while_scan'},
  {'sequence_index': 1,
   'mode': 'track_while_scan',
   'mode_id': 'radar_mode:an_apg_68:track_while_scan'},
  {'sequence_index': 2,
   'mode': 'track_while_scan',
   'mode_id': 'radar_mode:an_apg_68:track_while_scan'},
  {'sequence_index': 3,
   'mode': 'track_while_scan',
   'mode_id': 'radar_mode:an_apg_68:track_while_scan'},
  {'sequence_index': 4,
   'mode': 'track_while_scan',
   'mode_id': 'radar_mode:an_apg_68:track_while_scan'},
  {'sequence_index': 5,
   'mode': 'track_while_scan',
   'mode_id': 'radar_mode:an_apg_68:track_while_scan'},
  {'sequence_index': 6,
   'mode': 'track_while_scan',
   'mode_id': 'r

In [4]:
first_series = data["observation_series"][0]
inference_rows = observations_without_ground_truth(first_series)
{
    "series_id": first_series["series_id"],
    "observation_count": first_series["observation_count"],
    "duration_s": first_series["duration_s"],
    "mode_shift_sequence_indices": first_series["mode_shift_sequence_indices"],
    "ground_truth_modes": [truth["mode"] for truth in first_series["ground_truth_mode_sequence"][:10]],
    "inference_row_has_ground_truth": "ground_truth_label" in inference_rows[0],
    "first_observation_keys": list(first_series["observations"][0].keys()),
    "first_inference_row_keys": list(inference_rows[0].keys()),
}

{'series_id': 'esm_series_00001',
 'observation_count': 91,
 'duration_s': 44.988,
 'mode_shift_sequence_indices': [7, 10, 17, 39, 42, 65, 87],
 'ground_truth_modes': ['track_while_scan',
  'track_while_scan',
  'track_while_scan',
  'track_while_scan',
  'track_while_scan',
  'track_while_scan',
  'track_while_scan',
  'air_to_ground_mapping',
  'air_to_ground_mapping',
  'air_to_ground_mapping'],
 'inference_row_has_ground_truth': False,
 'first_observation_keys': ['observation_id',
  'series_id',
  'sequence_index',
  'elapsed_time_s',
  'timestamp_unix',
  'timestamp_iso8601',
  'sensor',
  'estimated_emitter_location',
  'approximate_kinematics',
  'esm_radar_parameters',
  'candidate_labels_from_shared_kg_features',
  'ground_truth_label'],
 'first_inference_row_keys': ['observation_id',
  'series_id',
  'sequence_index',
  'elapsed_time_s',
  'timestamp_unix',
  'timestamp_iso8601',
  'sensor',
  'estimated_emitter_location',
  'approximate_kinematics',
  'esm_radar_parameters',

In [5]:
output_path = Path("../generated/demo_esm_observation_series.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
output_path

WindowsPath('../generated/demo_esm_observation_series.json')